In [1]:
import io
import os
import pandas as pd
import paramiko

import matplotlib.pyplot as plt

In [2]:

# --- 提供いただいたヘッダー見本（タブ区切りテキスト） ---
raw_header_text = """Timestamp (end of interval)	Int Temp (ｰC)	Ext Temp (ｰC)	Pressure (hPa)	Rel Humidity (%)	Wiper count	Vbatt (V)	40m Wind Speed (m/s)	40m Wind Speed Dispersion (m/s)	40m Wind Speed min (m/s)	40m Wind Speed max (m/s)	40m Wind Direction (ｰ)	40m Z-wind (m/s)	40m Z-wind Dispersion (m/s)	40m CNR (dB)	40m CNR min (dB)	40m Dopp Spect Broad (m/s)	40m Data Availability (%)		60m Wind Speed (m/s)	60m Wind Speed Dispersion (m/s)	60m Wind Speed min (m/s)	60m Wind Speed max (m/s)	60m Wind Direction (ｰ)	60m Z-wind (m/s)	60m Z-wind Dispersion (m/s)	60m CNR (dB)	60m CNR min (dB)	60m Dopp Spect Broad (m/s)	60m Data Availability (%)		80m Wind Speed (m/s)	80m Wind Speed Dispersion (m/s)	80m Wind Speed min (m/s)	80m Wind Speed max (m/s)	80m Wind Direction (ｰ)	80m Z-wind (m/s)	80m Z-wind Dispersion (m/s)	80m CNR (dB)	80m CNR min (dB)	80m Dopp Spect Broad (m/s)	80m Data Availability (%)		100m Wind Speed (m/s)	100m Wind Speed Dispersion (m/s)	100m Wind Speed min (m/s)	100m Wind Speed max (m/s)	100m Wind Direction (ｰ)	100m Z-wind (m/s)	100m Z-wind Dispersion (m/s)	100m CNR (dB)	100m CNR min (dB)	100m Dopp Spect Broad (m/s)	100m Data Availability (%)		120m Wind Speed (m/s)	120m Wind Speed Dispersion (m/s)	120m Wind Speed min (m/s)	120m Wind Speed max (m/s)	120m Wind Direction (ｰ)	120m Z-wind (m/s)	120m Z-wind Dispersion (m/s)	120m CNR (dB)	120m CNR min (dB)	120m Dopp Spect Broad (m/s)	120m Data Availability (%)		140m Wind Speed (m/s)	140m Wind Speed Dispersion (m/s)	140m Wind Speed min (m/s)	140m Wind Speed max (m/s)	140m Wind Direction (ｰ)	140m Z-wind (m/s)	140m Z-wind Dispersion (m/s)	140m CNR (dB)	140m CNR min (dB)	140m Dopp Spect Broad (m/s)	140m Data Availability (%)		160m Wind Speed (m/s)	160m Wind Speed Dispersion (m/s)	160m Wind Speed min (m/s)	160m Wind Speed max (m/s)	160m Wind Direction (ｰ)	160m Z-wind (m/s)	160m Z-wind Dispersion (m/s)	160m CNR (dB)	160m CNR min (dB)	160m Dopp Spect Broad (m/s)	160m Data Availability (%)		180m Wind Speed (m/s)	180m Wind Speed Dispersion (m/s)	180m Wind Speed min (m/s)	180m Wind Speed max (m/s)	180m Wind Direction (ｰ)	180m Z-wind (m/s)	180m Z-wind Dispersion (m/s)	180m CNR (dB)	180m CNR min (dB)	180m Dopp Spect Broad (m/s)	180m Data Availability (%)		200m Wind Speed (m/s)	200m Wind Speed Dispersion (m/s)	200m Wind Speed min (m/s)	200m Wind Speed max (m/s)	200m Wind Direction (ｰ)	200m Z-wind (m/s)	200m Z-wind Dispersion (m/s)	200m CNR (dB)	200m CNR min (dB)	200m Dopp Spect Broad (m/s)	200m Data Availability (%)		220m Wind Speed (m/s)	220m Wind Speed Dispersion (m/s)	220m Wind Speed min (m/s)	220m Wind Speed max (m/s)	220m Wind Direction (ｰ)	220m Z-wind (m/s)	220m Z-wind Dispersion (m/s)	220m CNR (dB)	220m CNR min (dB)	220m Dopp Spect Broad (m/s)	220m Data Availability (%)		240m Wind Speed (m/s)	240m Wind Speed Dispersion (m/s)	240m Wind Speed min (m/s)	240m Wind Speed max (m/s)	240m Wind Direction (ｰ)	240m Z-wind (m/s)	240m Z-wind Dispersion (m/s)	240m CNR (dB)	240m CNR min (dB)	240m Dopp Spect Broad (m/s)	240m Data Availability (%)		260m Wind Speed (m/s)	260m Wind Speed Dispersion (m/s)	260m Wind Speed min (m/s)	260m Wind Speed max (m/s)	260m Wind Direction (ｰ)	260m Z-wind (m/s)	260m Z-wind Dispersion (m/s)	260m CNR (dB)	260m CNR min (dB)	260m Dopp Spect Broad (m/s)	260m Data Availability (%)		280m Wind Speed (m/s)	280m Wind Speed Dispersion (m/s)	280m Wind Speed min (m/s)	280m Wind Speed max (m/s)	280m Wind Direction (ｰ)	280m Z-wind (m/s)	280m Z-wind Dispersion (m/s)	280m CNR (dB)	280m CNR min (dB)	280m Dopp Spect Broad (m/s)	280m Data Availability (%)		300m Wind Speed (m/s)	300m Wind Speed Dispersion (m/s)	300m Wind Speed min (m/s)	300m Wind Speed max (m/s)	300m Wind Direction (ｰ)	300m Z-wind (m/s)	300m Z-wind Dispersion (m/s)	300m CNR (dB)	300m CNR min (dB)	300m Dopp Spect Broad (m/s)	300m Data Availability (%)"""

clean_headers = [c.strip() for c in raw_header_text.split("\t") if c.strip() != ""]

# --- SFTP接続設定 ---
sftp_host = "172.16.34.223"
sftp_user = "hayashi"
sftp_pass = "hayashi"
remote_dir = "/home/hayashi/kaiseki/SampleData/FREA_LiDAR/WLS71045_sta_files/"

# 1. ここで確実にSSHとSFTPの接続を行います
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(sftp_host, username=sftp_user, password=sftp_pass)
sftp = ssh.open_sftp()  # ここで 'sftp' が定義されます

df_list = []

try:
    # 2. 接続が確立された状態でループを回します
    for n in range(1, 29):
        date_str = f"{n:02d}"
        file_name = f"WLS71045_2025_02_{date_str}__00_10_00.sta"
        remote_path = os.path.join(remote_dir, file_name)

        print(f"ファイルを処理中: {file_name}")

        try:
            with sftp.open(remote_path, "rb") as f:
                binary_data = f.read()

            text_data = binary_data.decode("shift_jis", errors="ignore")

            df_raw = pd.read_csv(
                io.StringIO(text_data),
                sep="\s+",
                skiprows=42,
                header=None,
            )

            if df_raw.empty:
                print(f"⚠️ {file_name} はデータが空のためスキップします。")
                continue

            # 日時の結合
            combined_datetime = pd.to_datetime(
                df_raw[0].astype(str) + " " + df_raw[1].astype(str),
                errors="coerce",
            )

            df_clean = pd.DataFrame()
            df_clean[0] = combined_datetime

            # 残りのデータ列を結合
            df_data_part = df_raw.iloc[:, 2:]
            df_clean = pd.concat([df_clean, df_data_part], axis=1)
            df_clean.columns = range(df_clean.shape[1])

            # 列名のマッピング
            final_columns = []
            actual_cols = df_clean.shape[1]
            for i in range(actual_cols):
                if i < len(clean_headers):
                    final_columns.append(clean_headers[i])
                else:
                    final_columns.append(f"Extra_Column_{i}")
            df_clean.columns = final_columns

            df_list.append(df_clean)

        except FileNotFoundError:
            print(
                f"⚠️ ファイルが存在しません（スキップします）: {file_name}"
            )
        except Exception as file_error:
            print(f"❌ ファイル {file_name} の処理中にエラー: {file_error}")

    # すべてのファイルを縦結合
    if df_list:
        print("\n全ファイルを1つのDataFrameに統合中...")
        df_all = pd.concat(df_list, axis=0, ignore_index=True)

        print("\n🎉 全ファイルの統合・整形が完全に成功しました！")
        print("最終データ形状（合計行数, 列数）:", df_all.shape)
        print("\n--- データの最初の3行 ---")
        print(df_all.head(3))
    else:
        print("\n⚠️ 有効なデータが1つも見つかりませんでした。")

except Exception as e:
    print(f"\n❌ エラーが発生しました: {e}")

finally:
    # 3. 最後に安全にクローズします
    sftp.close()
    ssh.close()

ファイルを処理中: WLS71045_2025_02_01__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_02__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_03__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_04__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_05__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_06__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_07__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_08__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_09__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_10__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_11__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_12__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_13__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_14__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_15__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_16__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_17__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_18__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_19__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_20__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_21__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_22__00_10_00.sta
ファイルを処理中: WLS71045_2025_02_23__0

In [3]:
df=pd.read_pickle("Sample_Feb.pkl")

In [15]:
df

,Timestamp (end of interval),Int Temp (ｰC),Ext Temp (ｰC),Pressure (hPa),Rel Humidity (%),Wiper count,Vbatt (V),40m Wind Speed (m/s),40m Wind Speed Dispersion (m/s),40m Wind Speed min (m/s),...,300m Wind Speed Dispersion (m/s),300m Wind Speed min (m/s),300m Wind Speed max (m/s),300m Wind Direction (ｰ),300m Z-wind (m/s),300m Z-wind Dispersion (m/s),300m CNR (dB),300m CNR min (dB),300m Dopp Spect Broad (m/s),300m Data Availability (%)
0,2025-02-01 00:10:00,4.3,0.57,988.02,79.27,0,NaN,2.19,1.05,0.22,...,2.08,0.25,11.36,239.61,0.65,1.14,-12.24,-18.76,0.03,100.0
1,2025-02-01 00:20:00,4.6,0.60,988.03,78.57,0,NaN,1.35,0.85,0.03,...,1.94,1.12,10.34,224.48,0.49,1.62,-12.75,-18.50,0.02,100.0
2,2025-02-01 00:30:00,4.5,0.60,988.00,78.78,0,NaN,2.59,1.22,0.24,...,2.57,1.28,14.32,232.28,0.62,1.37,-10.37,-17.55,0.02,100.0
3,2025-02-01 00:40:00,3.7,0.60,987.88,79.00,0,NaN,3.26,1.33,0.68,...,1.94,1.58,10.54,221.47,-0.02,1.37,-11.40,-16.78,0.02,100.0
4,2025-02-01 00:50:00,3.6,0.55,987.95,79.65,0,NaN,2.89,1.15,0.71,...,2.03,0.99,11.22,220.39,0.26,1.18,-10.94,-15.76,0.02,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3997,2025-02-28 23:20:00,6.4,165.33,987.14,42.95,0,NaN,3.68,1.00,1.99,...,1.32,3.28,10.27,197.94,0.13,0.75,-19.61,-21.01,0.07,100.0
3998,2025-02-28 23:30:00,6.4,161.73,987.10,45.74,0,NaN,3.89,1.22,0.10,...,1.75,0.43,10.93,204.93,0.66,0.80,-19.55,-20.81,0.09,100.0
3999,2025-02-28 23:40:00,6.4,164.67,987.03,43.74,0,NaN,2.93,0.94,0.24,...,1.45,2.60,10.88,200.41,0.65,0.93,-19.53,-20.94,0.06,100.0
4000,2025-02-28 23:50:00,6.4,165.13,987.14,43.89,0,NaN,3.95,1.01,1.02,...,1.50,3.34,11.24,203.70,0.60,0.86,-19.58,-21.19,0.07,100.0


In [4]:
import numpy as np
import pandas as pd

ts = "Timestamp (end of interval)"
d = df.copy()
d[ts] = pd.to_datetime(d[ts])

# 30分ごとのグループ（00:10, 00:20, 00:30 → 00:30）
d["group"] = d[ts].dt.ceil("30min")

dirs = [c for c in d.columns if "Wind Direction" in c]
others = [c for c in d.columns if c not in dirs + [ts, "group"]]

# 風向以外：単純平均
out = d.groupby("group")[others].mean()

# 風向：sin/cos それぞれ平均 → 角度に戻す
for c in dirs:
    rad = np.deg2rad(d[c])
    s = np.sin(rad).groupby(d["group"]).mean()
    co = np.cos(rad).groupby(d["group"]).mean()
    out[c] = np.rad2deg(np.arctan2(s, co)) % 360

# 元の列順に戻して、Timestamp を列に戻す
out = out[[c for c in df.columns if c != ts]]
out = out.rename_axis(ts).reset_index()

In [7]:
out.to_pickle("Sample_30minave.pkl")

In [6]:
df

,Timestamp (end of interval),Int Temp (ｰC),Ext Temp (ｰC),Pressure (hPa),Rel Humidity (%),Wiper count,Vbatt (V),40m Wind Speed (m/s),40m Wind Speed Dispersion (m/s),40m Wind Speed min (m/s),...,300m Wind Speed Dispersion (m/s),300m Wind Speed min (m/s),300m Wind Speed max (m/s),300m Wind Direction (ｰ),300m Z-wind (m/s),300m Z-wind Dispersion (m/s),300m CNR (dB),300m CNR min (dB),300m Dopp Spect Broad (m/s),300m Data Availability (%)
0,2025-02-01 00:10:00,4.3,0.57,988.02,79.27,0,NaN,2.19,1.05,0.22,...,2.08,0.25,11.36,239.61,0.65,1.14,-12.24,-18.76,0.03,100.0
1,2025-02-01 00:20:00,4.6,0.60,988.03,78.57,0,NaN,1.35,0.85,0.03,...,1.94,1.12,10.34,224.48,0.49,1.62,-12.75,-18.50,0.02,100.0
2,2025-02-01 00:30:00,4.5,0.60,988.00,78.78,0,NaN,2.59,1.22,0.24,...,2.57,1.28,14.32,232.28,0.62,1.37,-10.37,-17.55,0.02,100.0
3,2025-02-01 00:40:00,3.7,0.60,987.88,79.00,0,NaN,3.26,1.33,0.68,...,1.94,1.58,10.54,221.47,-0.02,1.37,-11.40,-16.78,0.02,100.0
4,2025-02-01 00:50:00,3.6,0.55,987.95,79.65,0,NaN,2.89,1.15,0.71,...,2.03,0.99,11.22,220.39,0.26,1.18,-10.94,-15.76,0.02,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3997,2025-02-28 23:20:00,6.4,165.33,987.14,42.95,0,NaN,3.68,1.00,1.99,...,1.32,3.28,10.27,197.94,0.13,0.75,-19.61,-21.01,0.07,100.0
3998,2025-02-28 23:30:00,6.4,161.73,987.10,45.74,0,NaN,3.89,1.22,0.10,...,1.75,0.43,10.93,204.93,0.66,0.80,-19.55,-20.81,0.09,100.0
3999,2025-02-28 23:40:00,6.4,164.67,987.03,43.74,0,NaN,2.93,0.94,0.24,...,1.45,2.60,10.88,200.41,0.65,0.93,-19.53,-20.94,0.06,100.0
4000,2025-02-28 23:50:00,6.4,165.13,987.14,43.89,0,NaN,3.95,1.01,1.02,...,1.50,3.34,11.24,203.70,0.60,0.86,-19.58,-21.19,0.07,100.0
